In [ ]:
"""VERİ HAZIRLAMA — TAR → LOCAL → NPY (OPTİMİZE)=================================================167GB RAM + 80GB GPU + 235GB DiskMultiprocessing ile paralel okuma → max hız"""from google.colab import drivedrive.mount('/content/drive')import os, time, gcimport cv2import numpy as npfrom pathlib import Pathfrom multiprocessing import Pool, cpu_countARCHIVE = Path('/content/drive/MyDrive/archive')PURE_BG_DIR = ARCHIVE / 'Data_Subset_Autoencoders_Anomaly_Detectors/training/pure_background'TESTING_DIR = ARCHIVE / 'Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final'NPY_DIR = ARCHIVE / 'npy_cache'NPY_DIR.mkdir(exist_ok=True)TAR_BG = ARCHIVE / 'pure_bg.tar'TAR_TEST = ARCHIVE / 'test_final.tar'LOCAL_BG = Path('/content/data/pure_background')LOCAL_TEST = Path('/content/data/testing_final')FRAMES_PER_CLIP = 16WORKERS = min(cpu_count(), 8)print(f"CPU cores: {cpu_count()} | Workers: {WORKERS}\n")

In [ ]:
"""CR-AE PROTOTİP v5 — NPY'DEN YÜKLE + EĞİT + TEST=====================================================NPY dosyaları hazır → 30sn yükleme → eğitim → test → AUC"""import subprocesssubprocess.run(['pip', 'install', 'torch torchvision scikit-learn', '-q'])from google.colab import drivedrive.mount('/content/drive')import torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderimport time, random, gcimport numpy as npfrom pathlib import Pathfrom IPython.display import display, Image as IPImageimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}\n")SEED = 42; random.seed(SEED); torch.manual_seed(SEED)NPY_DIR = Path('/content/drive/MyDrive/archive/npy_cache')

In [ ]:
"""CR-AE ASIL MODEL EĞİTİMİ (v3)================================Google Colab — A100 80GB GPU, 167GB RAMDeğişiklikler (v1 → v3):  1. Val split: Rastgele → Ay/gün bazında stratified (data leakage yok)  2. Batch size: 16 → 32 (A100 tensor core optimal)  3. plt.savefig sıralama düzeltmesi  4. weight_decay CONFIG'e eklendi (reproducibility)  5. EER (Equal Error Rate) metriği eklendi  6. Seasonal Robustness Score — ay bazlı AUC (concept drift analizi)  7. Inference Latency ölçümü (tez hedefi: <30ms/frame)Mimari: CNN Encoder → Projected LSTM → CNN DecoderVeri: 1902 klip pure_background (128x128)Test: 478 klip (323N + 155A)NPY'den yükleme → 30sn"""import subprocesssubprocess.run(['pip', 'install', 'torch', 'torchvision', 'scikit-learn', '-q'])from google.colab import drivedrive.mount('/content/drive')import torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderimport time, random, gc, jsonimport numpy as npfrom pathlib import Pathfrom collections import defaultdictfrom IPython.display import display, Image as IPImageimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")if device.type == 'cuda':    print(f"GPU: {torch.cuda.get_device_name()}")    print(f"GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB\n")SEED = 42random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)NPY_DIR = Path('/content/drive/MyDrive/archive/npy_cache')SAVE_DIR = Path('/content/drive/MyDrive/archive/models')SAVE_DIR.mkdir(exist_ok=True)

In [ ]:
"""MODEL HATA AYIKLAMA VE GÖRSELLEŞTİRME (V4 - KESİN DİZİN YOLU)============================================================Google Colab — A100Düzeltme:- Orijinal .jpg dosyalarının kök dizini tam paylaşılan klasör ağacına  (Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final)  göre güncellendi."""from google.colab import drivedrive.mount('/content/drive')import torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import DataLoader, TensorDatasetimport numpy as npimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport matplotlib.image as mpimgfrom pathlib import Pathimport globfrom sklearn.metrics import precision_recall_curvefrom IPython.display import displaydevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}\n")ARCHIVE_DIR = Path('/content/drive/MyDrive/archive')NPY_DIR = ARCHIVE_DIR / 'npy_cache'MODEL_DIR = ARCHIVE_DIR / 'models'TESTING_FINAL_DIR = ARCHIVE_DIR / 'Data_Subset_Autoencoders_Anomaly_Detectors' / 'testing' / 'testing_final'

In [ ]:
"""crae_full_evaluation.py========================1. Winter + Summer TAR → RAM → test_all.tar → Drive2. Winter → Summer → All sırayla test (TAR'dan oku, 128x128 resize, NPY RAM'de tut)3. Her set için: AUC-ROC, F1, Precision, Recall, Confusion Matrix, grafikler4. Recall 90+ eşiği için ayrı confusion matrix (3 set yan yana)5. Grafikler + NPY'ler locale/RAM'de — Drive'a YAZILMAZ (onay beklenir)Düzeltmeler:- frame sıralama: hem frame000.jpg hem frame_000.jpg formatı desteklenir- ALL TAR prefix: winter/ ve summer/ alt klasörleri doğru tanınır"""from google.colab import drivedrive.mount('/content/drive')import torch, tarfile, shutil, subprocess, time, warningsimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import DataLoader, TensorDatasetimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.gridspec as gridspecfrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,                             confusion_matrix, precision_score, recall_score)from IPython.display import display, Image as IPImageimport cv2warnings.filterwarnings('ignore')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")

In [ ]:
import shutilfrom pathlib import PathARCHIVE   = Path('/content/drive/MyDrive/archive')LOCAL_OUT = Path('/content/eval_results')TEST_RESULTS = ARCHIVE / 'test_results'TEST_RESULTS.mkdir(parents=True, exist_ok=True)

In [ ]:
"""crae_error_map_roi_v2.py=========================all_results + npy_cache_ram'den direkt al (Drive I/O yok)WINTER / SUMMER / ALL'dan:  - 3 en yüksek MSE anomali (net insan)  - 3 en düşük MSE normal  (net arka plan)Her set benzersiz klip — ALL'da winter/summer'da seçilenler tekrar alınmazÇıktı: /content/eval_results/error_maps/ + Drive'a kopyalanır"""import torch, cv2, shutilimport torch.nn as nnimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.patches as patchesfrom pathlib import Pathdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")ARCHIVE   = Path('/content/drive/MyDrive/archive')OUT_DIR   = Path('/content/eval_results/error_maps')OUT_DIR.mkdir(parents=True, exist_ok=True)FRAME_SIZE = 128FRAME_IDX  = 8N_SELECT   = 3

## ***ÖNCESİNDE KISLA EĞİTİM YAPILDI, KIS TESTİ:0.85+ YAZ TESTİ:0.90+ AUC ELDE EDİLDİ***

In [ ]:
"""build_all_tars.py==================4 TAR olusturur:1. pure_bg_winter.tar   — training/pure_background/YYYYMMDD/clip_xxx/2. pure_bg_summer.tar   — training/pure_background_summer/YYYYMMDD/clip_xxx/3. test_winter.tar      — testing/testing_final_winter/YYYYMMDD/normal|anomaly/clip_xxx/4. test_summer.tar      — testing/testing_final_summer/YYYYMMDD/normal|anomaly/clip_xxx/Cikti: archive/all_final_tars_crae/Yontem: Drive -> /tmp paralel kopyala -> tar -> Drive"""import shutil, time, subprocess, gcfrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom tqdm import tqdmARCHIVE  = Path('/content/drive/MyDrive/archive')BASE_AE  = ARCHIVE / 'Data_Subset_Autoencoders_Anomaly_Detectors'OUT_DIR  = ARCHIVE / 'all_final_tars_crae'OUT_DIR.mkdir(parents=True, exist_ok=True)TMP      = Path('/tmp/tar_build')TMP.mkdir(parents=True, exist_ok=True)WORKERS  = 128DATASETS = [    {        'name'   : 'pure_bg_winter',        'src'    : BASE_AE / 'training/pure_background',        'tar'    : OUT_DIR / 'pure_bg_winter.tar',        'has_labels': False,

In [ ]:
"""build_npy_cache.py===================4 TAR'dan 128x128 NPY cache olusturur:1. pure_bg_winter_128.npy2. pure_bg_summer_128.npy3. test_winter_128.npy  + labels + names4. test_summer_128.npy  + labels + namesCikti: archive/final_npycache/GPU RAM cache: gpu_cache dict"""import subprocess, shutil, time, gc, cv2import numpy as npimport torchfrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom tqdm import tqdmARCHIVE    = Path('/content/drive/MyDrive/archive')TAR_DIR    = ARCHIVE / 'all_final_tars_crae'OUT_DIR    = ARCHIVE / 'final_npycache'OUT_DIR.mkdir(parents=True, exist_ok=True)TMP        = Path('/tmp/npy_build')TMP.mkdir(parents=True, exist_ok=True)WORKERS    = 128BATCH      = 64FRAME_SIZE = 128CLIP_LEN   = 16device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")if device.type == 'cuda':    print(f"GPU: {torch.cuda.get_device_name()}")gpu_cache = {}# ══════════════════════════════════════════════════════════════════════════════#  YARDIMCI# ══════════════════════════════════════════════════════════════════════════════def find_actual_root(tmp_dst):    """TAR icinde ekstra klasor olabilir, YYYYMMDD icerenin parent'ini bul."""    for p in sorted(tmp_dst.rglob('*')):        if p.is_dir() and p.name.isdigit() and len(p.name) == 8:            return p.parent    return tmp_dstdef read_clip(clip_dir):    frames = sorted(clip_dir.glob("frame*.jpg"),                    key=lambda p: int(''.join(filter(str.isdigit, p.stem))))    if len(frames) < CLIP_LEN: return None    imgs = []    for f in frames[:CLIP_LEN]:        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)        if img is None: return None        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE))        imgs.append(img.astype(np.float32) / 255.0)    return np.stack(imgs)def read_batch(clip_dirs):    results = [None] * len(clip_dirs)    with ThreadPoolExecutor(max_workers=WORKERS) as ex:        futs = {ex.submit(read_clip, c): i for i, c in enumerate(clip_dirs)}        for fut in as_completed(futs):            results[futs[fut]] = fut.result()    return resultsdef load_all_clips(root, has_labels):    clip_dirs = []    labels    = []    names     = []    day_names = []    for day_dir in sorted(root.iterdir()):        if not day_dir.is_dir() or not day_dir.name.isdigit(): continue        if has_labels:            for lbl_dir in sorted(day_dir.iterdir()):                if not lbl_dir.is_dir(): continue                lbl_val = 0 if lbl_dir.name == 'normal' else 1                for clip_dir in sorted(lbl_dir.iterdir()):                    if not clip_dir.is_dir(): continue                    clip_dirs.append(clip_dir)                    labels.append(lbl_val)                    names.append(clip_dir.name)                    day_names.append(day_dir.name)        else:            for clip_dir in sorted(day_dir.iterdir()):                if not clip_dir.is_dir(): continue                clip_dirs.append(clip_dir)                labels.append(0)                names.append(clip_dir.name)                day_names.append(day_dir.name)    print(f"  Toplam klip: {len(clip_dirs)}")    if has_labels:        n = sum(1 for l in labels if l == 0)        a = sum(1 for l in labels if l == 1)        print(f"  Normal: {n}  Anomali: {a}")    all_clips = []    valid_idx = []    for i in tqdm(range(0, len(clip_dirs), BATCH), desc="  Batch"):        batch   = clip_dirs[i:i+BATCH]        results = read_batch(batch)        for j, arr in enumerate(results):            if arr is not None:                all_clips.append(arr)                valid_idx.append(i+j)    if not all_clips:        raise ValueError("Hic klip okunamadi! Klasor yapisi kontrol edilmeli.")    clips_np   = np.stack(all_clips).astype(np.float16)    labels_np  = np.array([labels[i]    for i in valid_idx], dtype=np.int8)    names_np   = np.array([names[i]     for i in valid_idx], dtype=object)    days_np    = np.array([day_names[i] for i in valid_idx], dtype=object)    return clips_np, labels_np, names_np, days_np# ══════════════════════════════════════════════════════════════════════════════#  DATASET TANIMLARI# ══════════════════════════════════════════════════════════════════════════════DATASETS = [    {        'name'       : 'pure_bg_winter',        'tar'        : TAR_DIR / 'pure_bg_winter.tar',        'has_labels' : False,        'npy_clips'  : OUT_DIR / 'pure_bg_winter_128.npy',        'npy_labels' : None,        'npy_names'  : OUT_DIR / 'pure_bg_winter_names.npy',    },    {        'name'       : 'pure_bg_summer',        'tar'        : TAR_DIR / 'pure_bg_summer.tar',        'has_labels' : False,        'npy_clips'  : OUT_DIR / 'pure_bg_summer_128.npy',        'npy_labels' : None,        'npy_names'  : OUT_DIR / 'pure_bg_summer_names.npy',    },    {        'name'       : 'test_winter',        'tar'        : TAR_DIR / 'test_winter.tar',        'has_labels' : True,        'npy_clips'  : OUT_DIR / 'test_winter_128.npy',        'npy_labels' : OUT_DIR / 'test_winter_labels.npy',        'npy_names'  : OUT_DIR / 'test_winter_names.npy',    },    {        'name'       : 'test_summer',        'tar'        : TAR_DIR / 'test_summer.tar',        'has_labels' : True,        'npy_clips'  : OUT_DIR / 'test_summer_128.npy',        'npy_labels' : OUT_DIR / 'test_summer_labels.npy',        'npy_names'  : OUT_DIR / 'test_summer_names.npy',    },]# ══════════════════════════════════════════════════════════════════════════════#  ANA DONGU# ══════════════════════════════════════════════════════════════════════════════total_start = time.time()for ds in DATASETS:    name       = ds['name']    tar_path   = ds['tar']    has_labels = ds['has_labels']    tmp_dst    = TMP / name    print(f"\n{'='*60}")    print(f"{name}")    print(f"{'='*60}")    # NPY zaten varsa yukle    if ds['npy_clips'].exists():        print(f"  NPY zaten var, yukleniyor...")        t0     = time.time()        clips  = np.load(str(ds['npy_clips']))        labels = np.load(str(ds['npy_labels'])) if ds['npy_labels'] and ds['npy_labels'].exists() else None        names  = np.load(str(ds['npy_names']), allow_pickle=True)        print(f"  {clips.shape}  dtype={clips.dtype}  ({time.time()-t0:.0f}sn)")    else:        # ── TAR → /tmp ────────────────────────────────────────────────────────        if tmp_dst.exists() and len(list(tmp_dst.rglob('*.jpg'))) > 100:            print(f"  /tmp hazir")        else:            tmp_dst.mkdir(parents=True, exist_ok=True)            print(f"  TAR aciliyor: {tar_path.name}...")            t0 = time.time()            subprocess.run(f"tar -xf '{tar_path}' -C '{tmp_dst}'",                           shell=True, check=True)            print(f"  TAR: {time.time()-t0:.0f}sn")        # Gercek root'u bul        actual_root = find_actual_root(tmp_dst)        print(f"  Root: {actual_root.relative_to(TMP)}")        # ── Frame'leri oku ────────────────────────────────────────────────────        t0 = time.time()        clips, labels, names, day_names = load_all_clips(actual_root, has_labels)        print(f"  Okuma: {time.time()-t0:.0f}sn")        print(f"  Shape: {clips.shape}  dtype={clips.dtype}")        # ── NPY kaydet ────────────────────────────────────────────────────────        print(f"  NPY kaydediliyor...")        t0 = time.time()        np.save(str(ds['npy_clips']), clips)        np.save(str(ds['npy_names']), names)        if has_labels and ds['npy_labels']:            np.save(str(ds['npy_labels']), labels)        print(f"  Kaydedildi: {time.time()-t0:.0f}sn")        shutil.rmtree(str(tmp_dst), ignore_errors=True)        gc.collect()    # ── GPU RAM cache ─────────────────────────────────────────────────────────    print(f"  GPU RAM'e yukleniyor...")    t0 = time.time()    gpu_tensor = torch.tensor(        clips.astype(np.float32), dtype=torch.float16).to(device)    gpu_cache[name] = {        'clips' : gpu_tensor,        'labels': torch.tensor(labels.astype(np.int8)).to(device) if labels is not None else None,        'names' : names,    }    mb = gpu_tensor.element_size() * gpu_tensor.nelement() / 1e6    print(f"  GPU: {mb:.0f}MB  ({time.time()-t0:.0f}sn)")    del clips    gc.collect()# ══════════════════════════════════════════════════════════════════════════════#  OZET# ══════════════════════════════════════════════════════════════════════════════print(f"\n{'='*60}")print(f"TAMAMLANDI  ({(time.time()-total_start)/60:.1f}dk)")print(f"{'='*60}")print(f"\nNPY dosyalari:")for f in sorted(OUT_DIR.glob('*.npy')):    print(f"  {f.name:<45} {f.stat().st_size/1e6:>8.0f} MB")print(f"\nGPU RAM cache:")for k, v in gpu_cache.items():    shape = tuple(v['clips'].shape)    mb    = v['clips'].element_size()*v['clips'].nelement()/1e6    print(f"  {k:<28} {str(shape):<28} {mb:>8.0f} MB")if device.type == 'cuda':    alloc = torch.cuda.memory_allocated()/1e6    total = torch.cuda.get_device_properties(0).total_memory/1e6    print(f"\nGPU RAM: {alloc:.0f}MB / {total:.0f}MB")shutil.rmtree(str(TMP), ignore_errors=True)gc.collect()print("\nHazir! gpu_cache kullanima hazir.")

In [ ]:
"""crae_model1_winter_only.py===========================Model 1: Sadece Pure BG Winter ile egitimTest: Winter / Summer / AllCikti: models/crae_winter_only.pth       diff_technic_results/model1_winter_only_results.json"""import torch, torch.nn as nn, torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderimport time, random, gc, jsonimport numpy as npfrom pathlib import Pathfrom collections import defaultdictfrom IPython.display import display, Image as IPImageimport matplotlib; matplotlib.use('Agg')import matplotlib.pyplot as pltfrom sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,                              f1_score, confusion_matrix, classification_report)# ══════════════════════════════════════════════════════════════════════════════SEED = 42random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")if device.type == 'cuda':    print(f"GPU: {torch.cuda.get_device_name()}")    print(f"GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")MODEL_NAME = 'winter_only'NPY_DIR    = Path('/content/drive/MyDrive/archive/final_npycache')SAVE_DIR   = Path('/content/drive/MyDrive/archive/models')RES_DIR    = Path('/content/drive/MyDrive/archive/diff_technic_results')SAVE_DIR.mkdir(exist_ok=True)RES_DIR.mkdir(exist_ok=True)CONFIG = {    'frame_size'     : 128,    'frames_per_clip': 16,    'batch_size'     : 32,    'val_split'      : 0.15,    'epochs'         : 100,    'lr'             : 1e-3,    'min_lr'         : 1e-6,    'lstm_hidden'    : 256,    'lstm_layers'    : 2,    'dropout'        : 0.3,    'patience'       : 15,    'weight_decay'   : 1e-5,    'encoder_filters': [32, 64, 128, 256],}# ══════════════════════════════════════════════════════════════════════════════# ══════════════════════════════════════════════════════════════════════════════print("\n" + "="*60)print("VERI YUKLEME")print("="*60)t0 = time.time()# Egitimbg_clips = np.load(str(NPY_DIR/'pure_bg_winter_128.npy')).astype(np.float32)bg_names = np.load(str(NPY_DIR/'pure_bg_winter_names.npy'), allow_pickle=True)# Test setleritw_clips  = np.load(str(NPY_DIR/'test_winter_128.npy')).astype(np.float32)tw_labels = np.load(str(NPY_DIR/'test_winter_labels.npy'))tw_names  = np.load(str(NPY_DIR/'test_winter_names.npy'), allow_pickle=True)ts_clips  = np.load(str(NPY_DIR/'test_summer_128.npy')).astype(np.float32)ts_labels = np.load(str(NPY_DIR/'test_summer_labels.npy'))ts_names  = np.load(str(NPY_DIR/'test_summer_names.npy'), allow_pickle=True)# All = winter + summerall_clips  = np.concatenate([tw_clips,  ts_clips])all_labels = np.concatenate([tw_labels, ts_labels])all_names  = np.concatenate([tw_names,  ts_names])print(f"Pure BG Winter : {bg_clips.shape}")print(f"Test Winter    : {tw_clips.shape} (N:{(tw_labels==0).sum()} A:{(tw_labels==1).sum()})")print(f"Test Summer    : {ts_clips.shape} (N:{(ts_labels==0).sum()} A:{(ts_labels==1).sum()})")print(f"Test All       : {all_clips.shape} (N:{(all_labels==0).sum()} A:{(all_labels==1).sum()})")print(f"Yukleme: {time.time()-t0:.1f}sn")# ══════════════════════════════════════════════════════════════════════════════#  GUN BAZLI SPLIT# ══════════════════════════════════════════════════════════════════════════════def day_stratified_split(data, names, val_ratio=0.15, seed=42):    rng = random.Random(seed)    day_to_idx  = defaultdict(list)    month_to_days = defaultdict(set)    for i, name in enumerate(names):        d = str(name)[:8]; m = d[:6]        day_to_idx[d].append(i)        month_to_days[m].add(d)    val_idx = []; tr_idx = []    for month in sorted(month_to_days):        days = sorted(month_to_days[month])        n_val = max(1, round(len(days)*val_ratio))        shuf  = days[:]; rng.shuffle(shuf)        vd    = set(shuf[:n_val])        for d in days:            (val_idx if d in vd else tr_idx).extend(day_to_idx[d])    return np.array(tr_idx), np.array(val_idx)tr_idx, val_idx = day_stratified_split(bg_clips, bg_names, CONFIG['val_split'], SEED)train_data = bg_clips[tr_idx]val_data   = bg_clips[val_idx]print(f"\nTrain: {len(train_data)}  Val: {len(val_data)}")del bg_clips; gc.collect()# ══════════════════════════════════════════════════════════════════════════════#  DATASET# ══════════════════════════════════════════════════════════════════════════════class AugDataset(Dataset):    def __init__(self, data, aug=True):        self.data = data; self.aug = aug    def __len__(self): return len(self.data)    def __getitem__(self, i):        clip = self.data[i].copy()        if self.aug:            if random.random() > 0.5: clip = clip[:,:,::-1].copy()            clip = np.clip(clip + random.uniform(-0.1,0.1), 0, 1)            clip = np.clip(clip + np.random.normal(0,0.02,clip.shape).astype(np.float32), 0, 1)        return torch.tensor(clip).unsqueeze(1)class TestDS(Dataset):    def __init__(self, d, l): self.d=d; self.l=l    def __len__(self): return len(self.d)    def __getitem__(self, i):        return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]train_dl = DataLoader(AugDataset(train_data, True),  CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True)val_dl   = DataLoader(AugDataset(val_data,   False), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)tw_dl    = DataLoader(TestDS(tw_clips,  tw_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)ts_dl    = DataLoader(TestDS(ts_clips,  ts_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)all_dl   = DataLoader(TestDS(all_clips, all_labels), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)# ══════════════════════════════════════════════════════════════════════════════#  MODEL# ══════════════════════════════════════════════════════════════════════════════class CRAE(nn.Module):    def __init__(self, cfg):        super().__init__()        f = cfg['encoder_filters']        self.spatial = f[-1]*8*8        lstm_h = cfg['lstm_hidden']        drop   = cfg['dropout']        self.encoder = nn.Sequential(            nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),            nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),            nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),            nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),            nn.Dropout2d(drop),        )        self.proj   = nn.Sequential(nn.Linear(self.spatial, lstm_h), nn.LeakyReLU(0.2))        self.lstm   = nn.LSTM(lstm_h, lstm_h, cfg['lstm_layers'], batch_first=True,                              dropout=drop if cfg['lstm_layers']>1 else 0)        self.unproj = nn.Sequential(nn.Linear(lstm_h, self.spatial), nn.LeakyReLU(0.2))        self.decoder = nn.Sequential(            nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),        )    def forward(self, x):        B,T,C,H,W = x.shape        e = self.encoder(x.view(B*T,C,H,W))        e = self.proj(e.view(B*T,-1)).view(B,T,-1)        e,_ = self.lstm(e)        d = self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)        return self.decoder(d).view(B,T,1,H,W)model = CRAE(CONFIG).to(device)if hasattr(torch,'compile'):    try: model = torch.compile(model); print("torch.compile aktif")    except: passn_params = sum(p.numel() for p in model.parameters())print(f"Params: {n_params:,}")# ══════════════════════════════════════════════════════════════════════════════#  EGITIM# ══════════════════════════════════════════════════════════════════════════════criterion = nn.MSELoss()optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=CONFIG['min_lr'])best_val = float('inf'); pat = 0tl_h=[]; vl_h=[]; lr_h=[]t_start = time.time()save_path = SAVE_DIR / f'crae_{MODEL_NAME}.pth'print(f"\n{'='*60}\nEGITIM — {MODEL_NAME}\n{'='*60}")for ep in range(CONFIG['epochs']):    model.train()    el=nb=0    for batch in train_dl:        clips=batch.to(device); optimizer.zero_grad()        loss=criterion(model(clips),clips); loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)        optimizer.step(); el+=loss.item(); nb+=1    tl=el/max(nb,1); tl_h.append(tl)    model.eval()    vl=nv=0    with torch.no_grad():        for batch in val_dl:            clips=batch.to(device); vl+=criterion(model(clips),clips).item(); nv+=1    vl/=max(nv,1); vl_h.append(vl)    lr_now=optimizer.param_groups[0]['lr']; lr_h.append(lr_now)    scheduler.step()    if vl < best_val:        best_val=vl; pat=0        torch.save({'epoch':ep,'model_state':model.state_dict(),                    'val_loss':vl,'config':CONFIG,'model_name':MODEL_NAME}, str(save_path))    else: pat+=1    elapsed=time.time()-t_start    eta=elapsed/(ep+1)*(CONFIG['epochs']-ep-1)    print(f"  Ep{ep+1:>3} T:{tl:.6f} V:{vl:.6f} Best:{best_val:.6f} "          f"Pat:{pat}/{CONFIG['patience']} {elapsed/60:.1f}dk ETA:{eta/60:.1f}dk", flush=True)    if pat>=CONFIG['patience']:        print(f"  Early stopping ep{ep+1}"); breaktotal_time = time.time()-t_startprint(f"\nEgitim: {total_time/60:.1f}dk")# Egitim grafigifig,axes=plt.subplots(1,2,figsize=(14,5))axes[0].plot(tl_h,'b-',label='Train',alpha=0.7); axes[0].plot(vl_h,'r-',label='Val',alpha=0.7)axes[0].set_title(f'Egitim Egrisi — {MODEL_NAME}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)axes[1].plot(lr_h,'g-'); axes[1].set_title('LR Schedule'); axes[1].grid(True,alpha=0.3)plt.tight_layout()plt.savefig(f'/content/training_{MODEL_NAME}.png',dpi=150)plt.savefig(str(RES_DIR/f'training_{MODEL_NAME}.png'),dpi=150); plt.close()display(IPImage(f'/content/training_{MODEL_NAME}.png'))# ══════════════════════════════════════════════════════════════════════════════# ══════════════════════════════════════════════════════════════════════════════ck = torch.load(str(save_path)); model.load_state_dict(ck['model_state']); model.eval()def run_test(dl, tag):    scores=[]; labs=[]    with torch.no_grad():        for clips,lbls in dl:            clips=clips.to(device); out=model(clips)            for i in range(clips.shape[0]):                scores.append(torch.mean((clips[i]-out[i])**2).item())                labs.append(lbls[i].item())    scores=np.array(scores); labs=np.array(labs)    ns=scores[labs==0]; als=scores[labs==1]    auc=roc_auc_score(labs,scores)    prec,rec,thr=precision_recall_curve(labs,scores)    f1s=2*prec*rec/(prec+rec+1e-8)    opt_t=thr[np.argmax(f1s)]    preds=(scores>=opt_t).astype(int)    cm=confusion_matrix(labs,preds); tn,fp,fn,tp=cm.ravel()    fpr_r,tpr_r,thr_r=roc_curve(labs,scores)    fnr_r=1-tpr_r    eer_i=np.nanargmin(np.abs(fpr_r-fnr_r))    eer=float((fpr_r[eer_i]+fnr_r[eer_i])/2)    print(f"\n  {'='*50}")    print(f"  {tag}")    print(f"  {'='*50}")    print(f"  Normal MSE : {ns.mean():.6f} +- {ns.std():.6f}")    print(f"  Anomali MSE: {als.mean():.6f} +- {als.std():.6f}")    print(f"  AUC        : {auc:.4f}")    print(f"  EER        : {eer:.4f} ({eer*100:.2f}%)")    print(f"  F1         : {f1s.max():.4f}")    print(f"  Precision  : {tp/(tp+fp+1e-8):.4f}")    print(f"  Recall     : {tp/(tp+fn+1e-8):.4f}")    print(f"  Threshold  : {opt_t:.6f}")    print(f"  TP:{tp} FP:{fp} FN:{fn} TN:{tn}")    # ROC grafigi    fig,axes=plt.subplots(1,2,figsize=(12,5))    axes[0].plot(fpr_r,tpr_r,'b-',lw=2,label=f'AUC={auc:.4f}')    axes[0].plot([0,1],[0,1],'k--',alpha=0.3)    axes[0].set_title(f'ROC — {tag}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)    axes[1].hist(ns,bins=40,alpha=0.6,label=f'Normal ({len(ns)})',color='green')    axes[1].hist(als,bins=40,alpha=0.6,label=f'Anomali ({len(als)})',color='red')    axes[1].axvline(opt_t,color='black',ls='--',label=f'Thr={opt_t:.5f}')    axes[1].set_title(f'Score Dagilimi — {tag}'); axes[1].legend(); axes[1].grid(True,alpha=0.3)    plt.tight_layout()    fname=f'{MODEL_NAME}_{tag.lower().replace(" ","_")}'    plt.savefig(f'/content/{fname}.png',dpi=150)    plt.savefig(str(RES_DIR/f'{fname}.png'),dpi=150); plt.close()    return {        'tag':tag,'auc':float(auc),'eer':float(eer),        'f1':float(f1s.max()),'threshold':float(opt_t),        'precision':float(tp/(tp+fp+1e-8)),'recall':float(tp/(tp+fn+1e-8)),        'specificity':float(tn/(tn+fp+1e-8)),        'normal_mse':float(ns.mean()),'anomaly_mse':float(als.mean()),        'n_normal':int((labs==0).sum()),'n_anomaly':int((labs==1).sum()),        'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),    }print(f"\n{'='*60}\nTEST\n{'='*60}")r_winter = run_test(tw_dl,  'WINTER')r_summer = run_test(ts_dl,  'SUMMER')r_all    = run_test(all_dl, 'ALL')# ══════════════════════════════════════════════════════════════════════════════#  KAYDET# ══════════════════════════════════════════════════════════════════════════════results = {    'model_name'   : MODEL_NAME,    'training_data': 'pure_bg_winter',    'config'       : CONFIG,    'n_params'     : n_params,    'train_clips'  : len(train_data),    'val_clips'    : len(val_data),    'best_val_loss': float(best_val),    'best_epoch'   : int(ck['epoch'])+1,    'total_epochs' : len(tl_h),    'training_time_min': round(total_time/60,1),    'test_winter'  : r_winter,    'test_summer'  : r_summer,    'test_all'     : r_all,    'train_losses' : [float(x) for x in tl_h],    'val_losses'   : [float(x) for x in vl_h],}out_json = RES_DIR/f'model1_{MODEL_NAME}_results.json'with open(str(out_json),'w') as f: json.dump(results,f,indent=2)print(f"\n{'='*60}")print(f"OZET — Model 1: {MODEL_NAME}")print(f"{'='*60}")print(f"  {'Test Seti':<12} {'AUC':>8} {'EER':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")print(f"  {'-'*55}")for r in [r_winter, r_summer, r_all]:    print(f"  {r['tag']:<12} {r['auc']:>8.4f} {r['eer']:>8.4f} "          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f}")print(f"\n  Model : {save_path}")print(f"  JSON  : {out_json}")print("="*60)gc.collect()print("\nTamamlandi!")

In [ ]:
"""crae_model2_summer_only.py===========================Model 2: Sadece Pure BG Summer ile egitimTest: Winter / Summer / AllCikti: models/crae_summer_only.pth       diff_technic_results/model2_summer_only_results.json"""import torch, torch.nn as nn, torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderimport time, random, gc, jsonimport numpy as npfrom pathlib import Pathfrom collections import defaultdictfrom IPython.display import display, Image as IPImageimport matplotlib; matplotlib.use('Agg')import matplotlib.pyplot as pltfrom sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,                              confusion_matrix)# ══════════════════════════════════════════════════════════════════════════════SEED = 42random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)if torch.cuda.is_available(): torch.cuda.manual_seed(SEED)device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")if device.type == 'cuda':    print(f"GPU: {torch.cuda.get_device_name()}")    print(f"GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")MODEL_NAME = 'summer_only'NPY_DIR    = Path('/content/drive/MyDrive/archive/final_npycache')SAVE_DIR   = Path('/content/drive/MyDrive/archive/models')RES_DIR    = Path('/content/drive/MyDrive/archive/diff_technic_results')SAVE_DIR.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)CONFIG = {    'frame_size'     : 128,    'frames_per_clip': 16,    'batch_size'     : 32,    'val_split'      : 0.15,    'epochs'         : 100,    'lr'             : 1e-3,    'min_lr'         : 1e-6,    'lstm_hidden'    : 256,    'lstm_layers'    : 2,    'dropout'        : 0.3,    'patience'       : 15,    'weight_decay'   : 1e-5,    'encoder_filters': [32, 64, 128, 256],}# ══════════════════════════════════════════════════════════════════════════════#  VERI YUKLE# ══════════════════════════════════════════════════════════════════════════════print("\n" + "="*60)print("VERI YUKLEME")print("="*60)t0 = time.time()bg_clips = np.load(str(NPY_DIR/'pure_bg_summer_128.npy')).astype(np.float32)bg_names = np.load(str(NPY_DIR/'pure_bg_summer_names.npy'), allow_pickle=True)tw_clips  = np.load(str(NPY_DIR/'test_winter_128.npy')).astype(np.float32)tw_labels = np.load(str(NPY_DIR/'test_winter_labels.npy'))tw_names  = np.load(str(NPY_DIR/'test_winter_names.npy'), allow_pickle=True)ts_clips  = np.load(str(NPY_DIR/'test_summer_128.npy')).astype(np.float32)ts_labels = np.load(str(NPY_DIR/'test_summer_labels.npy'))ts_names  = np.load(str(NPY_DIR/'test_summer_names.npy'), allow_pickle=True)all_clips  = np.concatenate([tw_clips,  ts_clips])all_labels = np.concatenate([tw_labels, ts_labels])all_names  = np.concatenate([tw_names,  ts_names])print(f"Pure BG Summer : {bg_clips.shape}")print(f"Test Winter    : {tw_clips.shape} (N:{(tw_labels==0).sum()} A:{(tw_labels==1).sum()})")print(f"Test Summer    : {ts_clips.shape} (N:{(ts_labels==0).sum()} A:{(ts_labels==1).sum()})")print(f"Test All       : {all_clips.shape} (N:{(all_labels==0).sum()} A:{(all_labels==1).sum()})")print(f"Yukleme: {time.time()-t0:.1f}sn")# ══════════════════════════════════════════════════════════════════════════════#  GUN BAZLI SPLIT# ══════════════════════════════════════════════════════════════════════════════def day_stratified_split(data, names, val_ratio=0.15, seed=42):    rng = random.Random(seed)    day_to_idx = defaultdict(list); month_to_days = defaultdict(set)    for i, name in enumerate(names):        d=str(name)[:8]; m=d[:6]        day_to_idx[d].append(i); month_to_days[m].add(d)    val_idx=[]; tr_idx=[]    for month in sorted(month_to_days):        days=sorted(month_to_days[month])        n_val=max(1,round(len(days)*val_ratio))        shuf=days[:]; rng.shuffle(shuf); vd=set(shuf[:n_val])        for d in days:            (val_idx if d in vd else tr_idx).extend(day_to_idx[d])    return np.array(tr_idx), np.array(val_idx)tr_idx, val_idx = day_stratified_split(bg_clips, bg_names, CONFIG['val_split'], SEED)train_data = bg_clips[tr_idx]; val_data = bg_clips[val_idx]print(f"\nTrain: {len(train_data)}  Val: {len(val_data)}")del bg_clips; gc.collect()# ══════════════════════════════════════════════════════════════════════════════#  DATASET# ══════════════════════════════════════════════════════════════════════════════class AugDataset(Dataset):    def __init__(self, data, aug=True):        self.data=data; self.aug=aug    def __len__(self): return len(self.data)    def __getitem__(self, i):        clip=self.data[i].copy()        if self.aug:            if random.random()>0.5: clip=clip[:,:,::-1].copy()            clip=np.clip(clip+random.uniform(-0.1,0.1),0,1)            clip=np.clip(clip+np.random.normal(0,0.02,clip.shape).astype(np.float32),0,1)        return torch.tensor(clip).unsqueeze(1)class TestDS(Dataset):    def __init__(self, d, l): self.d=d; self.l=l    def __len__(self): return len(self.d)    def __getitem__(self, i): return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]train_dl = DataLoader(AugDataset(train_data,True),  CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True)val_dl   = DataLoader(AugDataset(val_data,  False), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)tw_dl    = DataLoader(TestDS(tw_clips,  tw_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)ts_dl    = DataLoader(TestDS(ts_clips,  ts_labels),  CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)all_dl   = DataLoader(TestDS(all_clips, all_labels), CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)# ══════════════════════════════════════════════════════════════════════════════#  MODEL# ══════════════════════════════════════════════════════════════════════════════class CRAE(nn.Module):    def __init__(self, cfg):        super().__init__()        f=cfg['encoder_filters']; self.spatial=f[-1]*8*8        lstm_h=cfg['lstm_hidden']; drop=cfg['dropout']        self.encoder=nn.Sequential(            nn.Conv2d(1,f[0],3,2,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),            nn.Conv2d(f[0],f[1],3,2,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),            nn.Conv2d(f[1],f[2],3,2,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),            nn.Conv2d(f[2],f[3],3,2,1), nn.BatchNorm2d(f[3]), nn.LeakyReLU(0.2),            nn.Dropout2d(drop),        )        self.proj  =nn.Sequential(nn.Linear(self.spatial,lstm_h), nn.LeakyReLU(0.2))        self.lstm  =nn.LSTM(lstm_h,lstm_h,cfg['lstm_layers'],batch_first=True,                            dropout=drop if cfg['lstm_layers']>1 else 0)        self.unproj=nn.Sequential(nn.Linear(lstm_h,self.spatial), nn.LeakyReLU(0.2))        self.decoder=nn.Sequential(            nn.ConvTranspose2d(f[3],f[2],3,2,1,1), nn.BatchNorm2d(f[2]), nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[2],f[1],3,2,1,1), nn.BatchNorm2d(f[1]), nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[1],f[0],3,2,1,1), nn.BatchNorm2d(f[0]), nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[0],1,3,2,1,1), nn.Sigmoid(),        )    def forward(self, x):        B,T,C,H,W=x.shape        e=self.encoder(x.view(B*T,C,H,W))        e=self.proj(e.view(B*T,-1)).view(B,T,-1)        e,_=self.lstm(e)        d=self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)        return self.decoder(d).view(B,T,1,H,W)model=CRAE(CONFIG).to(device)if hasattr(torch,'compile'):    try: model=torch.compile(model); print("torch.compile aktif")    except: passn_params=sum(p.numel() for p in model.parameters())print(f"Params: {n_params:,}")# ══════════════════════════════════════════════════════════════════════════════#  EGITIM# ══════════════════════════════════════════════════════════════════════════════criterion=nn.MSELoss()optimizer=optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])scheduler=optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_0=10,T_mult=2,eta_min=CONFIG['min_lr'])best_val=float('inf'); pat=0; tl_h=[]; vl_h=[]; lr_h=[]t_start=time.time()save_path=SAVE_DIR/f'crae_{MODEL_NAME}.pth'print(f"\n{'='*60}\nEGITIM — {MODEL_NAME}\n{'='*60}")for ep in range(CONFIG['epochs']):    model.train(); el=nb=0    for batch in train_dl:        clips=batch.to(device); optimizer.zero_grad()        loss=criterion(model(clips),clips); loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)        optimizer.step(); el+=loss.item(); nb+=1    tl=el/max(nb,1); tl_h.append(tl)    model.eval(); vl=nv=0    with torch.no_grad():        for batch in val_dl:            clips=batch.to(device); vl+=criterion(model(clips),clips).item(); nv+=1    vl/=max(nv,1); vl_h.append(vl)    lr_now=optimizer.param_groups[0]['lr']; lr_h.append(lr_now)    scheduler.step()    if vl<best_val:        best_val=vl; pat=0        torch.save({'epoch':ep,'model_state':model.state_dict(),                    'val_loss':vl,'config':CONFIG,'model_name':MODEL_NAME}, str(save_path))    else: pat+=1    elapsed=time.time()-t_start; eta=elapsed/(ep+1)*(CONFIG['epochs']-ep-1)    print(f"  Ep{ep+1:>3} T:{tl:.6f} V:{vl:.6f} Best:{best_val:.6f} "          f"Pat:{pat}/{CONFIG['patience']} {elapsed/60:.1f}dk ETA:{eta/60:.1f}dk", flush=True)    if pat>=CONFIG['patience']: print(f"  Early stopping ep{ep+1}"); breaktotal_time=time.time()-t_startprint(f"\nEgitim: {total_time/60:.1f}dk")fig,axes=plt.subplots(1,2,figsize=(14,5))axes[0].plot(tl_h,'b-',label='Train',alpha=0.7); axes[0].plot(vl_h,'r-',label='Val',alpha=0.7)axes[0].set_title(f'Egitim Egrisi — {MODEL_NAME}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)axes[1].plot(lr_h,'g-'); axes[1].set_title('LR Schedule'); axes[1].grid(True,alpha=0.3)plt.tight_layout()plt.savefig(f'/content/training_{MODEL_NAME}.png',dpi=150)plt.savefig(str(RES_DIR/f'training_{MODEL_NAME}.png'),dpi=150); plt.close()display(IPImage(f'/content/training_{MODEL_NAME}.png'))# ══════════════════════════════════════════════════════════════════════════════#  TEST# ══════════════════════════════════════════════════════════════════════════════ck=torch.load(str(save_path)); model.load_state_dict(ck['model_state']); model.eval()def run_test(dl, tag):    scores=[]; labs=[]    with torch.no_grad():        for clips,lbls in dl:            clips=clips.to(device); out=model(clips)            for i in range(clips.shape[0]):                scores.append(torch.mean((clips[i]-out[i])**2).item())                labs.append(lbls[i].item())    scores=np.array(scores); labs=np.array(labs)    ns=scores[labs==0]; als=scores[labs==1]    auc=roc_auc_score(labs,scores)    prec,rec,thr=precision_recall_curve(labs,scores)    f1s=2*prec*rec/(prec+rec+1e-8); opt_t=thr[np.argmax(f1s)]    preds=(scores>=opt_t).astype(int)    cm=confusion_matrix(labs,preds); tn,fp,fn,tp=cm.ravel()    fpr_r,tpr_r,thr_r=roc_curve(labs,scores); fnr_r=1-tpr_r    eer_i=np.nanargmin(np.abs(fpr_r-fnr_r))    eer=float((fpr_r[eer_i]+fnr_r[eer_i])/2)    print(f"\n  {'='*50}\n  {tag}\n  {'='*50}")    print(f"  Normal MSE : {ns.mean():.6f} +- {ns.std():.6f}")    print(f"  Anomali MSE: {als.mean():.6f} +- {als.std():.6f}")    print(f"  AUC: {auc:.4f}  EER: {eer:.4f}  F1: {f1s.max():.4f}")    print(f"  Prec: {tp/(tp+fp+1e-8):.4f}  Rec: {tp/(tp+fn+1e-8):.4f}  Thr: {opt_t:.6f}")    print(f"  TP:{tp} FP:{fp} FN:{fn} TN:{tn}")    fig,axes=plt.subplots(1,2,figsize=(12,5))    axes[0].plot(fpr_r,tpr_r,'b-',lw=2,label=f'AUC={auc:.4f}')    axes[0].plot([0,1],[0,1],'k--',alpha=0.3)    axes[0].set_title(f'ROC — {tag}'); axes[0].legend(); axes[0].grid(True,alpha=0.3)    axes[1].hist(ns,bins=40,alpha=0.6,label=f'Normal ({len(ns)})',color='green')    axes[1].hist(als,bins=40,alpha=0.6,label=f'Anomali ({len(als)})',color='red')    axes[1].axvline(opt_t,color='black',ls='--',label=f'Thr={opt_t:.5f}')    axes[1].set_title(f'Score Dagilimi — {tag}'); axes[1].legend(); axes[1].grid(True,alpha=0.3)    plt.tight_layout()    fname=f'{MODEL_NAME}_{tag.lower().replace(" ","_")}'    plt.savefig(f'/content/{fname}.png',dpi=150)    plt.savefig(str(RES_DIR/f'{fname}.png'),dpi=150); plt.close()    return {'tag':tag,'auc':float(auc),'eer':float(eer),'f1':float(f1s.max()),            'threshold':float(opt_t),'precision':float(tp/(tp+fp+1e-8)),            'recall':float(tp/(tp+fn+1e-8)),'specificity':float(tn/(tn+fp+1e-8)),            'normal_mse':float(ns.mean()),'anomaly_mse':float(als.mean()),            'n_normal':int((labs==0).sum()),'n_anomaly':int((labs==1).sum()),            'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn)}print(f"\n{'='*60}\nTEST\n{'='*60}")r_winter=run_test(tw_dl,  'WINTER')r_summer=run_test(ts_dl,  'SUMMER')r_all   =run_test(all_dl, 'ALL')results={'model_name':MODEL_NAME,'training_data':'pure_bg_summer','config':CONFIG,         'n_params':n_params,'train_clips':len(train_data),'val_clips':len(val_data),         'best_val_loss':float(best_val),'best_epoch':int(ck['epoch'])+1,         'total_epochs':len(tl_h),'training_time_min':round(total_time/60,1),         'test_winter':r_winter,'test_summer':r_summer,'test_all':r_all,         'train_losses':[float(x) for x in tl_h],'val_losses':[float(x) for x in vl_h]}out_json=RES_DIR/f'model2_{MODEL_NAME}_results.json'with open(str(out_json),'w') as f: json.dump(results,f,indent=2)print(f"\n{'='*60}\nOZET — Model 2: {MODEL_NAME}\n{'='*60}")print(f"  {'Test Seti':<12} {'AUC':>8} {'EER':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")print(f"  {'-'*55}")for r in [r_winter,r_summer,r_all]:    print(f"  {r['tag']:<12} {r['auc']:>8.4f} {r['eer']:>8.4f} "          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f}")print(f"\n  Model: {save_path}\n  JSON : {out_json}")gc.collect(); print("\nTamamlandi!")

In [ ]:
import torch, torch.nn as nnimport numpy as np, jsonfrom pathlib import Pathfrom collections import defaultdictimport matplotlib.pyplot as pltfrom IPython.display import display, Image as IPImagedevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')NPY_DIR  = Path('/content/drive/MyDrive/archive/final_npycache')RES_DIR  = Path('/content/drive/MyDrive/archive/diff_technic_results')SAVE_DIR = Path('/content/drive/MyDrive/archive/models/cc')CONFIG = {    'frame_size':128,'frames_per_clip':16,'batch_size':32,'val_split':0.15,    'epochs':100,'lr':1e-3,'min_lr':1e-6,'lstm_hidden':256,'lstm_layers':2,    'dropout':0.3,'patience':15,'weight_decay':1e-5,'encoder_filters':[32,64,128,256],}class CRAE(nn.Module):    def __init__(self, cfg):        super().__init__()        f=cfg['encoder_filters']; self.spatial=f[-1]*8*8        lstm_h=cfg['lstm_hidden']; drop=cfg['dropout']        self.encoder=nn.Sequential(            nn.Conv2d(1,f[0],3,2,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),            nn.Conv2d(f[0],f[1],3,2,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),            nn.Conv2d(f[1],f[2],3,2,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),            nn.Conv2d(f[2],f[3],3,2,1),nn.BatchNorm2d(f[3]),nn.LeakyReLU(0.2),            nn.Dropout2d(drop),        )        self.proj  =nn.Sequential(nn.Linear(self.spatial,lstm_h),nn.LeakyReLU(0.2))        self.lstm  =nn.LSTM(lstm_h,lstm_h,cfg['lstm_layers'],batch_first=True,                            dropout=drop if cfg['lstm_layers']>1 else 0)        self.unproj=nn.Sequential(nn.Linear(lstm_h,self.spatial),nn.LeakyReLU(0.2))        self.decoder=nn.Sequential(            nn.ConvTranspose2d(f[3],f[2],3,2,1,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[2],f[1],3,2,1,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[1],f[0],3,2,1,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[0],1,3,2,1,1),nn.Sigmoid(),        )    def forward(self, x):        B,T,C,H,W=x.shape        e=self.encoder(x.view(B*T,C,H,W))        e=self.proj(e.view(B*T,-1)).view(B,T,-1)        e,_=self.lstm(e)        d=self.unproj(e.reshape(B*T,-1)).view(B*T,CONFIG['encoder_filters'][-1],8,8)        return self.decoder(d).view(B,T,1,H,W)ck = torch.load(str(SAVE_DIR/'crae_winter_only.pth'), map_location=device)model = CRAE(CONFIG).to(device)state = {k.replace('_orig_mod.',''):v for k,v in ck['model_state'].items()}model.load_state_dict(state)model.eval()print("Model yuklendi")

In [ ]:
"""crae_error_analysis.py=======================Kis eğitimli + Yaz eğitimli model için:- Winter test + Summer test hata analizi- FN (kaçırılan anomali) + FP (yanlış alarm)- Tam Drive yolu: testing_final_winter/YYYYMMDD/label/clip- İki modelde ortak yanlış sınıflandırmalar"""import torch, torch.nn as nnimport numpy as np, jsonfrom pathlib import Pathfrom collections import defaultdictfrom torch.utils.data import Dataset, DataLoaderimport matplotlib; matplotlib.use('Agg')import matplotlib.pyplot as pltfrom sklearn.metrics import roc_auc_score, precision_recall_curvefrom IPython.display import display, Image as IPImagedevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"Device: {device}")NPY_DIR   = Path('/content/drive/MyDrive/archive/final_npycache')SAVE_DIR  = Path('/content/drive/MyDrive/archive/models')RES_DIR   = Path('/content/drive/MyDrive/archive/diff_technic_results')BASE_AE   = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/testing')TEST_WIN  = BASE_AE / 'testing_final_winter'TEST_SUM  = BASE_AE / 'testing_final_summer'RES_DIR.mkdir(exist_ok=True)CONFIG = {    'frame_size':128,'frames_per_clip':16,'batch_size':32,'val_split':0.15,    'epochs':100,'lr':1e-3,'min_lr':1e-6,'lstm_hidden':256,'lstm_layers':2,    'dropout':0.3,'patience':15,'weight_decay':1e-5,'encoder_filters':[32,64,128,256],}

In [ ]:
import torch, torch.nn as nnimport numpy as npfrom pathlib import Pathfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.metrics import (roc_auc_score, precision_recall_curve,                              roc_curve, confusion_matrix, classification_report)import cv2from concurrent.futures import ThreadPoolExecutor, as_completedfrom tqdm import tqdmimport gcdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')TEST_WIN = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final_winter')TEST_SUM = Path('/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors/testing/testing_final_summer')SAVE_DIR = Path('/content/drive/MyDrive/archive/models')RES_DIR  = Path('/content/drive/MyDrive/archive/diff_technic_results')WORKERS    = 128BATCH      = 64FRAME_SIZE = 128CLIP_LEN   = 16# ── Model ─────────────────────────────────────────────────────────────────────class CRAE(nn.Module):    def __init__(self, cfg):        super().__init__()        f=cfg['encoder_filters']; self.spatial=f[-1]*8*8        lstm_h=cfg['lstm_hidden']; drop=cfg['dropout']        self.encoder=nn.Sequential(            nn.Conv2d(1,f[0],3,2,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),            nn.Conv2d(f[0],f[1],3,2,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),            nn.Conv2d(f[1],f[2],3,2,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),            nn.Conv2d(f[2],f[3],3,2,1),nn.BatchNorm2d(f[3]),nn.LeakyReLU(0.2),            nn.Dropout2d(drop),        )        self.proj  =nn.Sequential(nn.Linear(self.spatial,lstm_h),nn.LeakyReLU(0.2))        self.lstm  =nn.LSTM(lstm_h,lstm_h,cfg['lstm_layers'],batch_first=True,                            dropout=drop if cfg['lstm_layers']>1 else 0)        self.unproj=nn.Sequential(nn.Linear(lstm_h,self.spatial),nn.LeakyReLU(0.2))        self.decoder=nn.Sequential(            nn.ConvTranspose2d(f[3],f[2],3,2,1,1),nn.BatchNorm2d(f[2]),nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[2],f[1],3,2,1,1),nn.BatchNorm2d(f[1]),nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[1],f[0],3,2,1,1),nn.BatchNorm2d(f[0]),nn.LeakyReLU(0.2),            nn.ConvTranspose2d(f[0],1,3,2,1,1),nn.Sigmoid(),        )    def forward(self, x):        B,T,C,H,W=x.shape        e=self.encoder(x.view(B*T,C,H,W))        e=self.proj(e.view(B*T,-1)).view(B,T,-1)        e,_=self.lstm(e)        d=self.unproj(e.reshape(B*T,-1)).view(B*T,self.spatial//64,8,8)        return self.decoder(d).view(B,T,1,H,W)def load_model(path):    ck=torch.load(str(path),map_location=device)    cfg=ck.get('config',{'encoder_filters':[32,64,128,256],                          'lstm_hidden':256,'lstm_layers':2,'dropout':0.3})    m=CRAE(cfg).to(device)    state={k.replace('_orig_mod.',''):v for k,v in ck['model_state'].items()}    m.load_state_dict(state); m.eval()    return m, cfgdef read_clip(clip_dir):    frames=sorted(clip_dir.glob("frame*.jpg"),                  key=lambda p: int(''.join(filter(str.isdigit,p.stem))))    if len(frames)<CLIP_LEN: return None    imgs=[]    for fr in frames[:CLIP_LEN]:        img=cv2.imread(str(fr),cv2.IMREAD_GRAYSCALE)        if img is None: return None        img=cv2.resize(img,(FRAME_SIZE,FRAME_SIZE))        imgs.append(img.astype(np.float32)/255.0)    return np.stack(imgs)def load_test_set(test_root, tag):    clip_dirs=[]; labels=[]; names=[]    for day_dir in sorted(test_root.iterdir()):        if not day_dir.is_dir() or not day_dir.name.isdigit(): continue        for lbl_dir in sorted(day_dir.iterdir()):            if not lbl_dir.is_dir(): continue            lbl_name=lbl_dir.name.lower()            if 'anomal' in lbl_name:   lbl_val=1            elif 'normal' in lbl_name: lbl_val=0            else: continue            for clip_dir in sorted(lbl_dir.iterdir()):                if not clip_dir.is_dir(): continue                clip_dirs.append(clip_dir)                labels.append(lbl_val)                names.append(clip_dir.name)    n_norm=sum(1 for l in labels if l==0)    n_anom=sum(1 for l in labels if l==1)    print(f"  {tag}: {len(clip_dirs)} klip  Normal:{n_norm}  Anomali:{n_anom}")    clips_list=[None]*len(clip_dirs)    for i in tqdm(range(0,len(clip_dirs),BATCH),desc=f"  {tag}"):        batch=clip_dirs[i:i+BATCH]        with ThreadPoolExecutor(max_workers=WORKERS) as ex:            futs={ex.submit(read_clip,d):j for j,d in enumerate(batch)}            for fut in as_completed(futs):                clips_list[i+futs[fut]]=fut.result()    valid=[(c,l,n) for c,l,n in zip(clips_list,labels,names) if c is not None]    clips=np.stack([v[0] for v in valid]).astype(np.float32)    labs =np.array([v[1] for v in valid])    nms  =np.array([v[2] for v in valid])    return clips, labs, nmsclass TestDS(Dataset):    def __init__(self,d,l): self.d=d; self.l=l    def __len__(self): return len(self.d)    def __getitem__(self,i): return torch.tensor(self.d[i]).unsqueeze(1), self.l[i]import matplotlib; matplotlib.use('Agg')import matplotlib.pyplot as pltfrom IPython.display import display, Image as IPImageimport jsondef evaluate_full(model, clips, labels, tag, model_name):    ds=TestDS(clips,labels)    dl=DataLoader(ds,batch_size=64,shuffle=False,num_workers=2)    scores=[]    with torch.no_grad():        for bc,_ in dl:            bc=bc.to(device); out=model(bc)            for i in range(bc.shape[0]):                scores.append(torch.mean((bc[i]-out[i])**2).item())    scores=np.array(scores)    auc=roc_auc_score(labels,scores)    prec,rec,thr=precision_recall_curve(labels,scores)    f1s=2*prec*rec/(prec+rec+1e-8)    opt_t=thr[np.argmax(f1s)]    preds=(scores>=opt_t).astype(int)    cm=confusion_matrix(labels,preds)    tn,fp,fn,tp=cm.ravel()    fpr_r,tpr_r,thr_r=roc_curve(labels,scores)    fnr_r=1-tpr_r    eer_i=np.nanargmin(np.abs(fpr_r-fnr_r))    eer=float((fpr_r[eer_i]+fnr_r[eer_i])/2)    ns=scores[labels==0]; als=scores[labels==1]    print(f"\n  {'='*55}")    print(f"  {model_name} — {tag}")    print(f"  {'='*55}")    print(f"  AUC        : {auc:.4f}")    print(f"  EER        : {eer:.4f} ({eer*100:.2f}%)")    print(f"  F1         : {f1s.max():.4f}")    print(f"  Precision  : {tp/(tp+fp+1e-8):.4f}")    print(f"  Recall     : {tp/(tp+fn+1e-8):.4f}")    print(f"  Specificity: {tn/(tn+fp+1e-8):.4f}")    print(f"  Threshold  : {opt_t:.6f}")    print(f"  Normal MSE : {ns.mean():.6f} +- {ns.std():.6f}")    print(f"  Anomali MSE: {als.mean():.6f} +- {als.std():.6f}")    print(f"\n  Confusion Matrix:")    print(f"              Pred_N  Pred_A")    print(f"  Actual_N    {tn:>6}  {fp:>6}")    print(f"  Actual_A    {fn:>6}  {tp:>6}")    print(f"\n{classification_report(labels,preds,target_names=['Normal','Anomali'],digits=4)}")    # Grafik    fig,axes=plt.subplots(1,3,figsize=(18,5))    fig.suptitle(f'{model_name} — {tag}',fontsize=12,fontweight='bold')    axes[0].plot(fpr_r,tpr_r,'b-',lw=2,label=f'AUC={auc:.4f}')    axes[0].plot([0,1],[0,1],'k--',alpha=0.3)    axes[0].set_title('ROC'); axes[0].legend(); axes[0].grid(True,alpha=0.3)    axes[1].hist(ns,bins=40,alpha=0.6,label=f'Normal ({len(ns)})',color='green')    axes[1].hist(als,bins=40,alpha=0.6,label=f'Anomali ({len(als)})',color='red')    axes[1].axvline(opt_t,color='black',ls='--',label=f'Thr={opt_t:.5f}')    axes[1].set_title('Score Dagilimi'); axes[1].legend(); axes[1].grid(True,alpha=0.3)    im=axes[2].imshow(cm,cmap='Blues')    axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])    axes[2].set_xticklabels(['Normal','Anomali'])    axes[2].set_yticklabels(['Normal','Anomali'])    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')    axes[2].set_title('Confusion Matrix')    for i in range(2):        for j in range(2):            axes[2].text(j,i,f'{cm[i,j]}',ha='center',va='center',                         fontsize=16,color='white' if cm[i,j]>cm.max()/2 else 'black')    plt.tight_layout()    fname=f'{model_name}_{tag.lower()}_final'    plt.savefig(f'/content/{fname}.png',dpi=130)    plt.savefig(str(RES_DIR/f'{fname}.png'),dpi=130); plt.close()    display(IPImage(f'/content/{fname}.png'))    return {        'tag':tag,'model':model_name,'auc':float(auc),'eer':float(eer),        'f1':float(f1s.max()),'threshold':float(opt_t),        'precision':float(tp/(tp+fp+1e-8)),'recall':float(tp/(tp+fn+1e-8)),        'specificity':float(tn/(tn+fp+1e-8)),        'normal_mse':float(ns.mean()),'anomaly_mse':float(als.mean()),        'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),        'n_normal':int((labels==0).sum()),'n_anomaly':int((labels==1).sum()),    }# ══════════════════════════════════════════════════════════════════════════════print("Veri yukleniyor...")tw_clips,tw_labels,tw_names = load_test_set(TEST_WIN,"Winter")ts_clips,ts_labels,ts_names = load_test_set(TEST_SUM,"Summer")all_clips  = np.concatenate([tw_clips,ts_clips])all_labels = np.concatenate([tw_labels,ts_labels])# Model 1print("\n" + "="*60)print("MODEL 1: crae_winter_only.pth")print("="*60)m1,_ = load_model(SAVE_DIR/'crae_winter_only.pth')r_m1_w   = evaluate_full(m1, tw_clips,  tw_labels,  'WINTER', 'M1_WinterOnly')r_m1_s   = evaluate_full(m1, ts_clips,  ts_labels,  'SUMMER', 'M1_WinterOnly')r_m1_all = evaluate_full(m1, all_clips, all_labels, 'ALL',    'M1_WinterOnly')del m1; gc.collect(); torch.cuda.empty_cache()# Model 2print("\n" + "="*60)print("MODEL 2: crae_summer_only.pth")print("="*60)m2,_ = load_model(SAVE_DIR/'crae_summer_only.pth')r_m2_w   = evaluate_full(m2, tw_clips,  tw_labels,  'WINTER', 'M2_SummerOnly')r_m2_s   = evaluate_full(m2, ts_clips,  ts_labels,  'SUMMER', 'M2_SummerOnly')r_m2_all = evaluate_full(m2, all_clips, all_labels, 'ALL',    'M2_SummerOnly')del m2; gc.collect(); torch.cuda.empty_cache()print(f"\n{'='*65}")print("FINAL OZET")print(f"{'='*65}")print(f"  {'':20} {'AUC':>8} {'EER':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")print(f"  {'-'*55}")for r in [r_m1_w,r_m1_s,r_m1_all,r_m2_w,r_m2_s,r_m2_all]:    label=f"{r['model']}_{r['tag']}"    print(f"  {label:<20} {r['auc']:>8.4f} {r['eer']:>8.4f} "          f"{r['f1']:>8.4f} {r['precision']:>8.4f} {r['recall']:>8.4f}")# JSONall_results = {    'm1_winter':r_m1_w,'m1_summer':r_m1_s,'m1_all':r_m1_all,    'm2_winter':r_m2_w,'m2_summer':r_m2_s,'m2_all':r_m2_all,}with open(str(RES_DIR/'final_model_results.json'),'w') as f:    json.dump(all_results,f,indent=2)print(f"\nJSON: {RES_DIR/'final_model_results.json'}")print("Tamamlandi!")

In [ ]:
"""CR-AE Test — NPY'siz, doğrudan Drive'dan==========================================Preprocess: build_npy_cache.py ile BİREBİR AYNI  - cv2 grayscale + /255  - (B, T, 1, H, W) input  - skor = mean MSE (tüm T,C,H,W)  - LeakyReLU(0.2), Dropout2d encoder'da"""import os, cv2import numpy as npimport torchimport torch.nn as nnimport matplotlib.pyplot as pltfrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom sklearn.metrics import (roc_curve, auc, precision_recall_curve,                             confusion_matrix, ConfusionMatrixDisplay)import warnings; warnings.filterwarnings("ignore")

In [ ]:
"""Dataset & Model Analiz Raporu==============================Her iki model için:- Eğitim config ve hiperparametreler- Hangi günlerden kaç klip kullanıldı- Eğitim/val split dağılımı- Test seti kalite analizi (gün, sınıf, klip dağılımı)- Score dağılımı istatistikleri- Tüm çıktı metin tabanlı (print)"""import os, jsonimport numpy as npimport torchfrom pathlib import Pathfrom collections import defaultdictBASE     = Path("/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors")NPY_DIR  = Path("/content/drive/MyDrive/archive/final_npycache")RES_DIR  = BASE / "Testing_Results"MODELS   = {    "M1_WinterOnly": BASE / "models/crae_winter_only.pth",    "M2_SummerOnly": BASE / "models/crae_summer_only.pth",    "M3_Mixed":      BASE / "models/crae_mixed.pth",}TESTS = {    "Winter": BASE / "testing/testing_final_winter",    "Summer": BASE / "testing/testing_final_summer",}ANOMALY_CLASSES = {"anomaly", "trespassing", "loitering", "object_abandonment"}SEP  = "=" * 70SEP2 = "-" * 70

In [ ]:
"""CR-AE Test — NPY'siz, doğrudan Drive'dan==========================================Preprocess: build_npy_cache.py ile BİREBİR AYNI  - cv2 grayscale + /255  - (B, T, 1, H, W) input  - skor = mean MSE (tüm T,C,H,W)  - LeakyReLU(0.2), Dropout2d encoder'da"""import os, cv2import numpy as npimport torchimport torch.nn as nnimport matplotlib.pyplot as pltfrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom sklearn.metrics import (roc_curve, auc, precision_recall_curve,                             confusion_matrix, ConfusionMatrixDisplay)import warnings; warnings.filterwarnings("ignore")

In [ ]:
import osimport numpy as npfrom pathlib import Pathfrom collections import defaultdictBASE    = Path("/content/drive/MyDrive/archive/Data_Subset_Autoencoders_Anomaly_Detectors")NPY_DIR = BASE / "final_npycache"TESTS   = {    "Winter": BASE / "testing/testing_final_winter",    "Summer": BASE / "testing/testing_final_summer",}SEP  = "=" * 60SEP2 = "-" * 60print(SEP)print("  BÖLÜM 4 — KLİP KALİTE ANALİZİ")print(SEP)for test_name, test_path in TESTS.items():    print(f"\n  [Test: {test_name}]")    print(SEP2)    frame_counts, empty, short = [], [], []    for day in sorted(os.listdir(test_path)):        dp = test_path / day        if not dp.is_dir(): continue        for cls in os.listdir(dp):            cp = dp / cls            if not cp.is_dir(): continue            for cd in os.listdir(cp):                full = cp / cd                if not full.is_dir(): continue                frames = [f for f in os.listdir(full)                          if f.lower().endswith((".jpg",".jpeg",".png"))]                n = len(frames)                frame_counts.append(n)                if n == 0:   empty.append(f"{day}/{cls}/{cd}")                elif n < 16: short.append(f"{day}/{cls}/{cd} ({n} frame)")    arr = np.array(frame_counts)    print(f"  Toplam klip        : {len(arr)}")    print(f"  Frame min/max/ort  : {arr.min()} / {arr.max()} / {arr.mean():.1f}")    print(f"  Tam 16 frame       : {int((arr==16).sum())}  ({(arr==16).mean()*100:.1f}%)")    print(f"  Boş klip           : {len(empty)}")    print(f"  Kısa klip (<16)    : {len(short)}")    for c in empty[:5]:  print(f"    [BOŞ]  {c}")    for c in short[:5]:  print(f"    [KISA] {c}")print(f"\n{SEP}")print("  BÖLÜM 6 — EĞİTİM / TEST GÜN ÖRTÜŞME KONTROLÜ")print(SEP)

In [ ]:
"""crae_finetune_m1_summer.py===========================M1_WinterOnly → pure_bg_summer fine-tuneSadece eğitim. Test ayrı scriptte yapılacak.Çıktı: models/crae_winter_finetuned.pth"""import torch, torch.nn as nn, torch.optim as optimfrom torch.utils.data import Dataset, DataLoaderimport time, random, gc, jsonimport numpy as npfrom pathlib import Pathfrom collections import defaultdictfrom IPython.display import display, Image as IPImageimport matplotlib; matplotlib.use('Agg')import matplotlib.pyplot as plt

In [ ]:
"""crae_test_finetuned2.py======================crae_winter_finetuned2.pth modelini winter ve summer test setleriyle test eder.Veri: testing_final_winter / testing_final_summer klasörlerinden paralel yükleme.Çıktı: Testing_Results/finetuning_model/"""import torch, torch.nn as nnfrom torch.utils.data import Dataset, DataLoaderimport time, json, cv2import numpy as npfrom pathlib import Pathfrom concurrent.futures import ThreadPoolExecutor, as_completedfrom IPython.display import display, Image as IPImageimport matplotlib; matplotlib.use('Agg')import matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, confusion_matrix